### Baseline SEM

In [ ]:
# ============================================================
# SEM for Governmental Capacity and Disaster Recovery Outcomes
# Dataset: all_state_local_fund_latent_var_v2.csv
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from semopy import Model, calc_stats

# -----------------------------
# 1. Load data
# -----------------------------
file_path = "all_state_local_fund_latent_var_4_v2.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# -----------------------------
# 2. Create state/local dummy
#    state_level = 1 if Grantee is a 2-letter state abbreviation
# -----------------------------
df["state_level"] = df["Grantee"].astype(str).str.fullmatch(r"[A-Z]{2}").astype(int)

# Optional readable label
df["gov_level"] = np.where(df["state_level"] == 1, "state", "local")

print("\nGovernment level counts:")
print(df["gov_level"].value_counts(dropna=False))

# -----------------------------
# 3. Create SEM variables
#    avg_employment and avg_payroll are already population-normalized
# -----------------------------
eps = 1e-6

# Workload per staff
df["programs_per_staff"] = df["Num_Program"] / (df["avg_employment"] * df['E_TOTPOP'] + eps)
df["disasters_per_staff"] = df["Num_Disaster"] / (df["avg_employment"] * df['E_TOTPOP'] + eps)

# Reverse-code so higher = stronger capacity
df["rev_programs_per_staff"] = -df["programs_per_staff"]
df["rev_disasters_per_staff"] = -df["disasters_per_staff"]

# Reverse-code recovery timeliness indicators so higher = better
df["rev_Duration_of_completion"] = -df["Duration_of_completion"]
df["rev_Average_Duration_Program_Completion"] = -df["Average_Duration_Program_Completion"]
# df["rev_Quarter_variance"] = -df["Quarter_by_quarter_variance_expended"]

# -----------------------------
# 4. Select variables for SEM
# -----------------------------
sem_vars = [
    # Government capacity indicators
    "avg_employment",
    "avg_payroll",
    "rev_programs_per_staff",
    "rev_disasters_per_staff",

    # Recovery performance indicators
    "Ratio_disbursed_to_obligated",
    "Ratio_expended_to_disbursed",
    "Ratio_obligated_funds_fully_expended",
    "Ratio_Program_Completed",

    # Recovery timeliness indicators
    # "Timeliness",
    "rev_Duration_of_completion",
    "rev_Average_Duration_Program_Completion",
    # "rev_Quarter_variance",

    # Controls
    "state_level",
    "E_TOTPOP",
    "SPL_THEME1",
    "SPL_THEME2",
    "SPL_THEME3",
    "SPL_THEME4"
]

# Keep only needed columns
data = df[["Grantee"] + sem_vars].copy()

# -----------------------------
# 5. Handle missing values
#    Here we use listwise deletion for SEM
# -----------------------------
before_n = len(data)
data = data.dropna().copy()
after_n = len(data)

print(f"\nRows before dropping missing: {before_n}")
print(f"Rows after dropping missing:  {after_n}")
print(f"Rows removed:                 {before_n - after_n}")

# -----------------------------
# 6. Standardize continuous variables
#    Keep state_level as 0/1, do not standardize it
# -----------------------------
continuous_vars = [
    "avg_employment",
    "avg_payroll",
    "rev_programs_per_staff",
    "rev_disasters_per_staff",
    "Ratio_disbursed_to_obligated",
    "Ratio_expended_to_disbursed",
    "Ratio_obligated_funds_fully_expended",
    "Ratio_Program_Completed",
    "rev_Duration_of_completion",
    "rev_Average_Duration_Program_Completion",
    "E_TOTPOP",
    "SPL_THEME1",
    "SPL_THEME2",
    "SPL_THEME3",
    "SPL_THEME4"
]

scaler = StandardScaler()
data_z = data.copy()
data_z[[f"z_{c}" for c in continuous_vars]] = scaler.fit_transform(data[continuous_vars])

# Final SEM dataframe
sem_data = data_z[[
    "Grantee",
    "state_level",
    "z_avg_employment",
    "z_avg_payroll",
    "z_rev_programs_per_staff",
    "z_rev_disasters_per_staff",
    "z_Ratio_disbursed_to_obligated",
    "z_Ratio_expended_to_disbursed",
    "z_Ratio_obligated_funds_fully_expended",
    "z_Ratio_Program_Completed",
    "z_rev_Duration_of_completion",
    "z_rev_Average_Duration_Program_Completion",
    "z_E_TOTPOP",
    "z_SPL_THEME1",
    "z_SPL_THEME2",
    "z_SPL_THEME3",
    "z_SPL_THEME4"
]].copy()

print("\nFinal SEM data shape:", sem_data.shape)

# -----------------------------
# 7. Optional: inspect descriptive stats
# -----------------------------
desc = sem_data.describe().T
print("\nDescriptive statistics:")
print(desc)

# Save descriptive stats
desc.to_csv("baseline_sem_4/baseline_sem_descriptive_statistics.csv")

# -----------------------------
# 8. SEM model specification
# -----------------------------
model_desc = """
# -------------------------
# Measurement model
# -------------------------
GovCapacity =~ z_avg_employment + z_avg_payroll + z_rev_programs_per_staff + z_rev_disasters_per_staff

RecoveryPerformance =~ z_Ratio_disbursed_to_obligated + z_Ratio_expended_to_disbursed + z_Ratio_obligated_funds_fully_expended + z_Ratio_Program_Completed

RecoveryTimeliness =~ z_rev_Duration_of_completion + z_rev_Average_Duration_Program_Completion

# -------------------------
# Structural model
# -------------------------
RecoveryPerformance ~ GovCapacity + state_level + z_E_TOTPOP + z_SPL_THEME1 + z_SPL_THEME2 + z_SPL_THEME3 + z_SPL_THEME4
RecoveryTimeliness  ~ GovCapacity + state_level + z_E_TOTPOP + z_SPL_THEME1 + z_SPL_THEME2 + z_SPL_THEME3 + z_SPL_THEME4

# Allow the two recovery constructs to covary
RecoveryPerformance ~~ RecoveryTimeliness
"""

print("\nSEM model:")
print(model_desc)

# -----------------------------
# 9. Fit SEM
# -----------------------------
model = Model(model_desc)
result = model.fit(sem_data.drop(columns=["Grantee"]))

print("\nModel fitting result:")
print(result)

# -----------------------------
# 10. Extract parameter estimates
# -----------------------------
estimates = model.inspect(std_est=True)
print("\nParameter estimates:")
print(estimates)

# Save parameter estimates
estimates.to_csv("baseline_sem_4/baseline_sem_parameter_estimates.csv", index=False)

# -----------------------------
# 11. Model fit statistics
# -----------------------------
stats = calc_stats(model)

print("\nModel fit statistics:")
print(stats.T)

# Save fit statistics
stats.T.to_csv("baseline_sem_4/baseline_sem_fit_statistics.csv")

# -----------------------------
# 12. Extract key structural paths
# -----------------------------
key_paths = estimates[
    (estimates["op"] == "~") &
    (estimates["lval"].isin(["RecoveryPerformance", "RecoveryTimeliness"]))
].copy()

print("\nKey structural paths:")
print(key_paths)

key_paths.to_csv("baseline_sem_4/baseline_sem_key_structural_paths.csv", index=False)

# -----------------------------
# 13. Extract standardized factor loadings
# -----------------------------
factor_loadings = estimates[estimates["op"] == "=~"].copy()
print("\nFactor loadings:")
print(factor_loadings)

factor_loadings.to_csv("baseline_sem_4/baseline_sem_factor_loadings.csv", index=False)

# -----------------------------
# 14. Simple interpretation helper
# -----------------------------
def interpret_path(row):
    outcome = row["lval"]
    predictor = row["rval"]
    est = row["Est. Std"] if "Est. Std" in row.index else np.nan
    pval = row["p-value"] if "p-value" in row.index else np.nan

    if pd.isna(est):
        return f"{predictor} -> {outcome}: standardized estimate unavailable."

    direction = "positive" if est > 0 else "negative"
    sig = "statistically significant" if pd.notna(pval) and pval < 0.05 else "not statistically significant"

    return f"{predictor} -> {outcome}: {direction} association (std. beta = {est:.3f}), {sig} (p = {pval:.4g})."

print("\nInterpretation of key structural paths:")
for _, row in key_paths.iterrows():
    print("-", interpret_path(row))

# -----------------------------
# 15. Save SEM-ready dataset
# -----------------------------
sem_data.to_csv("baseline_sem_4/baseline_sem_ready_dataset.csv", index=False)

print("\nSaved files:")
print(" - sem_ready_dataset.csv")
print(" - sem_descriptive_statistics.csv")
print(" - sem_parameter_estimates.csv")
print(" - sem_fit_statistics.csv")
print(" - sem_key_structural_paths.csv")
print(" - sem_factor_loadings.csv")

Dataset shape: (573, 19)

Columns:
['Grantee', 'Ratio_disbursed_to_obligated', 'Ratio_expended_to_disbursed', 'Timeliness', 'Duration_of_completion', 'Ratio_obligated_funds_fully_expended', 'Quarter_by_quarter_variance_expended', 'Num_Disaster', 'Num_Program', 'Ratio_Program_Completed', 'Average_Duration_Program_Completion', 'E_TOTPOP', 'avg_employment', 'avg_payroll', 'SPL_THEME1', 'SPL_THEME2', 'SPL_THEME3', 'SPL_THEME4', 'SPL_THEMES']

Government level counts:
gov_level
local    543
state     30
Name: count, dtype: int64

Rows before dropping missing: 573
Rows after dropping missing:  573
Rows removed:                 0

Final SEM data shape: (573, 17)

Descriptive statistics:
                                           count          mean       std  \
state_level                                573.0  5.235602e-02  0.222939   
z_avg_employment                           573.0 -6.200198e-17  1.000874   
z_avg_payroll                              573.0 -1.240040e-17  1.000874   
z_rev_p

#### Two-factor SEM

In [ ]:
# ============================================================
# Two-Factor SEM for Governmental Capacity and Recovery Outcomes
# Dataset: all_state_local_fund_latent_var_v2.csv
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from semopy import Model, calc_stats

# -----------------------------
# 1. Load data
# -----------------------------
file_path = "all_state_local_fund_latent_var_4_v2.csv"
df = pd.read_csv(file_path)

# -----------------------------
# 2. Create state/local dummy
# -----------------------------
df["state_level"] = df["Grantee"].astype(str).str.fullmatch(r"[A-Z]{2}").astype(int)
df["gov_level"] = np.where(df["state_level"] == 1, "state", "local")

# -----------------------------
# 3. Create derived variables
#    avg_employment and avg_payroll already normalized by population
# -----------------------------
eps = 1e-6

df["programs_per_staff"] = df["Num_Program"] / (df["avg_employment"] * df['E_TOTPOP'] + eps)
df["disasters_per_staff"] = df["Num_Disaster"] / (df["avg_employment"] * df['E_TOTPOP'] + eps)

# Reverse-code burden so higher = stronger capacity
df["rev_programs_per_staff"] = -df["programs_per_staff"]
df["rev_disasters_per_staff"] = -df["disasters_per_staff"]

# Reverse-code timeliness-related indicators so higher = better
df["rev_Duration_of_completion"] = -df["Duration_of_completion"]
df["rev_Average_Duration_Program_Completion"] = -df["Average_Duration_Program_Completion"]
# df["rev_Quarter_variance"] = -df["Quarter_by_quarter_variance_expended"]

# -----------------------------
# 4. Keep required variables
# -----------------------------
sem_vars = [
    "avg_employment",
    "avg_payroll",
    "rev_programs_per_staff",
    "rev_disasters_per_staff",
    "Ratio_disbursed_to_obligated",
    "Ratio_expended_to_disbursed",
    "Ratio_obligated_funds_fully_expended",
    "Ratio_Program_Completed",
    "rev_Duration_of_completion",
    "rev_Average_Duration_Program_Completion",
    "E_TOTPOP",
    "state_level",
    "SPL_THEME1",
    "SPL_THEME2",
    "SPL_THEME3",
    "SPL_THEME4"
]

data = df[["Grantee"] + sem_vars].copy()

# -----------------------------
# 5. Drop missing values
# -----------------------------
before_n = len(data)
data = data.dropna().copy()
after_n = len(data)

print(f"Rows before dropping missing: {before_n}")
print(f"Rows after dropping missing:  {after_n}")
print(f"Rows removed:                 {before_n - after_n}")

# -----------------------------
# 6. Standardize continuous variables
# -----------------------------
continuous_vars = [
    "avg_employment",
    "avg_payroll",
    "rev_programs_per_staff",
    "rev_disasters_per_staff",
    "Ratio_disbursed_to_obligated",
    "Ratio_expended_to_disbursed",
    "Ratio_obligated_funds_fully_expended",
    "Ratio_Program_Completed",
    "rev_Duration_of_completion",
    "rev_Average_Duration_Program_Completion",
    "E_TOTPOP",
    "SPL_THEME1",
    "SPL_THEME2",
    "SPL_THEME3",
    "SPL_THEME4"
]

scaler = StandardScaler()
data_z = data.copy()
data_z[[f"z_{c}" for c in continuous_vars]] = scaler.fit_transform(data[continuous_vars])

sem_data = data_z[[
    "Grantee",
    "state_level",
    "z_avg_employment",
    "z_avg_payroll",
    "z_rev_programs_per_staff",
    "z_rev_disasters_per_staff",
    "z_Ratio_disbursed_to_obligated",
    "z_Ratio_expended_to_disbursed",
    "z_Ratio_obligated_funds_fully_expended",
    "z_Ratio_Program_Completed",
    "z_rev_Duration_of_completion",
    "z_rev_Average_Duration_Program_Completion",
    "z_E_TOTPOP",
    "z_SPL_THEME1",
    "z_SPL_THEME2",
    "z_SPL_THEME3",
    "z_SPL_THEME4"
]].copy()

# -----------------------------
# 7. Model specification
# -----------------------------
model_desc_2factor = """
# Measurement model
AdminResources =~ z_avg_employment + z_avg_payroll
AdminBurdenCapacity =~ z_rev_programs_per_staff + z_rev_disasters_per_staff

RecoveryPerformance =~ z_Ratio_disbursed_to_obligated + z_Ratio_expended_to_disbursed + z_Ratio_obligated_funds_fully_expended + z_Ratio_Program_Completed

RecoveryTimeliness =~ z_rev_Duration_of_completion + z_rev_Average_Duration_Program_Completion

# Structural model
RecoveryPerformance ~ AdminResources + AdminBurdenCapacity + state_level + z_E_TOTPOP + z_SPL_THEME1 + z_SPL_THEME2 + z_SPL_THEME3 + z_SPL_THEME4
RecoveryTimeliness  ~ AdminResources + AdminBurdenCapacity + state_level + z_E_TOTPOP + z_SPL_THEME1 + z_SPL_THEME2 + z_SPL_THEME3 + z_SPL_THEME4

# Covariances
AdminResources ~~ AdminBurdenCapacity
RecoveryPerformance ~~ RecoveryTimeliness
"""

print(model_desc_2factor)

# -----------------------------
# 8. Fit model
# -----------------------------
model_2factor = Model(model_desc_2factor)
result_2factor = model_2factor.fit(sem_data.drop(columns=["Grantee"]))

print("\nModel fitting result:")
print(result_2factor)

# -----------------------------
# 9. Extract estimates
# -----------------------------
estimates_2factor = model_2factor.inspect(std_est=True)
print("\nParameter estimates:")
print(estimates_2factor)

# -----------------------------
# 10. Fit statistics
# -----------------------------
stats_2factor = calc_stats(model_2factor)
print("\nModel fit statistics:")
print(stats_2factor.T)

# -----------------------------
# 11. Save outputs
# -----------------------------
sem_data.to_csv("baseline_sem_4/sem_2factor_ready_dataset.csv", index=False)
estimates_2factor.to_csv("baseline_sem_4/sem_2factor_parameter_estimates.csv", index=False)
stats_2factor.T.to_csv("baseline_sem_4/sem_2factor_fit_statistics.csv")

# -----------------------------
# 12. Structural paths only
# -----------------------------
key_paths_2factor = estimates_2factor[
    (estimates_2factor["op"] == "~") &
    (estimates_2factor["lval"].isin(["RecoveryPerformance", "RecoveryTimeliness"]))
].copy()

print("\nKey structural paths:")
print(key_paths_2factor)

key_paths_2factor.to_csv("baseline_sem_4/sem_2factor_key_structural_paths.csv", index=False)

# -----------------------------
# 13. Factor loadings only
# -----------------------------
factor_loadings_2factor = estimates_2factor[
    estimates_2factor["op"] == "=~"
].copy()

print("\nFactor loadings:")
print(factor_loadings_2factor)

factor_loadings_2factor.to_csv("baseline_sem_4/sem_2factor_factor_loadings.csv", index=False)

# -----------------------------
# 14. Simple interpretation helper
# -----------------------------
def interpret_path(row):
    outcome = row["lval"]
    predictor = row["rval"]
    est = row["Est. Std"] if "Est. Std" in row.index else np.nan
    pval = row["p-value"] if "p-value" in row.index else np.nan

    if pd.isna(est):
        return f"{predictor} -> {outcome}: standardized estimate unavailable."

    direction = "positive" if est > 0 else "negative"
    sig = "statistically significant" if pd.notna(pval) and pval < 0.05 else "not statistically significant"

    return f"{predictor} -> {outcome}: {direction} association (std. beta = {est:.3f}), {sig} (p = {pval:.4g})."

print("\nInterpretation of key structural paths:")
for _, row in key_paths_2factor.iterrows():
    print("-", interpret_path(row))

print("\nSaved files:")
print(" - sem_2factor_ready_dataset.csv")
print(" - sem_2factor_parameter_estimates.csv")
print(" - sem_2factor_fit_statistics.csv")
print(" - sem_2factor_key_structural_paths.csv")
print(" - sem_2factor_factor_loadings.csv")

Rows before dropping missing: 573
Rows after dropping missing:  573
Rows removed:                 0

# Measurement model
AdminResources =~ z_avg_employment + z_avg_payroll
AdminBurdenCapacity =~ z_rev_programs_per_staff + z_rev_disasters_per_staff

RecoveryPerformance =~ z_Ratio_disbursed_to_obligated + z_Ratio_expended_to_disbursed + z_Ratio_obligated_funds_fully_expended + z_Ratio_Program_Completed

RecoveryTimeliness =~ z_rev_Duration_of_completion + z_rev_Average_Duration_Program_Completion

# Structural model
RecoveryPerformance ~ AdminResources + AdminBurdenCapacity + state_level + z_E_TOTPOP + z_SPL_THEME1 + z_SPL_THEME2 + z_SPL_THEME3 + z_SPL_THEME4
RecoveryTimeliness  ~ AdminResources + AdminBurdenCapacity + state_level + z_E_TOTPOP + z_SPL_THEME1 + z_SPL_THEME2 + z_SPL_THEME3 + z_SPL_THEME4

# Covariances
AdminResources ~~ AdminBurdenCapacity
RecoveryPerformance ~~ RecoveryTimeliness


Model fitting result:
Name of objective: MLW
Optimization method: SLSQP
Optimization succes

###  Comparison

In [ ]:
fit1 = pd.read_csv("baseline_sem/baseline_sem_fit_statistics.csv", index_col=0)
fit2 = pd.read_csv("baseline_sem/sem_2factor_fit_statistics.csv", index_col=0)

comparison = pd.concat(
    [fit1.rename(columns={fit1.columns[0]: "OneFactor"}),
     fit2.rename(columns={fit2.columns[0]: "TwoFactor"})],
    axis=1
)

print(comparison)
comparison.to_csv("baseline_sem_4/sem_model_comparison.csv")

                 OneFactor    TwoFactor
DoF             101.000000    98.000000
DoF Baseline    126.000000   126.000000
chi2            741.720473   469.819942
chi2 p-value      0.000000     0.000000
chi2 Baseline  4506.238794  4506.238794
CFI               0.853725     0.915114
GFI               0.835401     0.895740
AGFI              0.794659     0.865952
NFI               0.835401     0.895740
TLI               0.817518     0.890861
RMSEA             0.105311     0.081443
AIC              67.411098    74.360140
BIC             219.692098   239.693797
LogLik            1.294451     0.819930
